# 00 · Environment Check

Run this notebook once after setting up the environment. It verifies that:

1. the kernel is the project's `.venv`
2. PyTorch can use the GPU (CUDA)
3. PyTorch Geometric works: GCN / GAT / GraphSAGE layers, heterogeneous graphs, `NeighborLoader`, GNNExplainer
4. a protein language model (ESM-2, via Hugging Face) can embed a protein sequence on the GPU
5. the remaining libraries (pandas, Biopython, NetworkX, PyVis, Streamlit, ...) import correctly
6. `tqdm` progress bars show up

**Kernel:** top-right **Select Kernel → Python Environments → `.venv`**, then **Run All**.

In [1]:
import sys
import warnings
from pathlib import Path

# ipywidgets is not installed on purpose (VS Code fails to load the first widget),
# so tqdm shows text progress bars - hide its "IProgress not found" hint
warnings.filterwarnings("ignore", message="IProgress not found")

print("Python :", sys.version.split()[0])
print("Kernel :", sys.executable)
assert ".venv" in sys.executable, "Wrong kernel - select the project's .venv (top-right 'Select Kernel')"

# Project root, whether the notebook runs from the root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent
print("Project root:", ROOT)
for folder in ["data/raw", "data/processed", "notebooks", "src", "models", "results"]:
    print(f"  {folder:15s}", "ok" if (ROOT / folder).is_dir() else "MISSING")

Python : 3.11.4
Kernel : c:\Users\rajdeep\Desktop\Capstone Project_GNN\.venv\Scripts\python.exe
Project root: c:\Users\rajdeep\Desktop\Capstone Project_GNN
  data/raw        ok
  data/processed  ok
  notebooks       ok
  src             ok
  models          ok
  results         ok


## 1. PyTorch + GPU

In [2]:
import torch

print("torch          :", torch.__version__)
print("CUDA (build)   :", torch.version.cuda)
print("CUDA available :", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    gpu = torch.cuda.get_device_properties(0)
    print("GPU            :", gpu.name)
    print(f"VRAM           : {gpu.total_memory / 1024**3:.1f} GB")
    a = torch.randn(4096, 4096, device=device)
    print("Matrix multiply on GPU:", tuple((a @ a).shape))
else:
    print("WARNING: GPU not visible - everything will run on the CPU")

torch          : 2.14.1+cu130
CUDA (build)   : 13.0
CUDA available : True
GPU            : NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM           : 6.0 GB
Matrix multiply on GPU: (4096, 4096)


## 2. PyTorch Geometric

A toy protein-protein interaction (PPI) graph: 4 proteins, 3 interactions
(each interaction is stored in both directions), 16 features per protein.

In [3]:
import torch_geometric
import torch_geometric.typing
from torch_geometric.data import Data
from torch_geometric.nn import GATConv, GCNConv, SAGEConv

print("torch_geometric:", torch_geometric.__version__)
print("pyg-lib        :", torch_geometric.typing.WITH_PYG_LIB)

edge_index = torch.tensor([[0, 1, 1, 2, 2, 3],
                           [1, 0, 2, 1, 3, 2]])
ppi = Data(x=torch.randn(4, 16), edge_index=edge_index)
print(ppi)

ppi_gpu = ppi.to(device)
for conv in [GCNConv(16, 8), GATConv(16, 8, heads=2), SAGEConv(16, 8)]:
    out = conv.to(device)(ppi_gpu.x, ppi_gpu.edge_index)
    print(f"  {type(conv).__name__:8s} -> output {tuple(out.shape)} on {out.device}")

c:\Users\rajdeep\Desktop\Capstone Project_GNN\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


torch_geometric: 2.8.1
pyg-lib        : True
Data(x=[4, 16], edge_index=[2, 6])
  GCNConv  -> output (4, 8) on cuda:0
  GATConv  -> output (4, 16) on cuda:0
  SAGEConv -> output (4, 8) on cuda:0


A heterogeneous graph (protein + disease nodes), and mini-batch neighbour sampling
with `NeighborLoader` (needs pyg-lib), which GraphSAGE uses on large graphs.

In [4]:
from torch_geometric.data import HeteroData
from torch_geometric.loader import NeighborLoader

graph = HeteroData()
graph["protein"].x = torch.randn(4, 16)
graph["disease"].x = torch.randn(2, 8)
graph["protein", "interacts", "protein"].edge_index = edge_index
graph["protein", "associated_with", "disease"].edge_index = torch.tensor([[0, 3],
                                                                         [0, 1]])
print(graph)

loader = NeighborLoader(ppi, num_neighbors=[2, 2], batch_size=2)
print("NeighborLoader batch:", next(iter(loader)))

HeteroData(
  protein={ x=[4, 16] },
  disease={ x=[2, 8] },
  (protein, interacts, protein)={ edge_index=[2, 6] },
  (protein, associated_with, disease)={ edge_index=[2, 2] }
)
NeighborLoader batch: Data(x=[4, 16], edge_index=[2, 5], n_id=[4], e_id=[5], num_sampled_nodes=[3], num_sampled_edges=[2], input_id=[2], batch_size=2)


GNNExplainer: which interactions mattered for a protein's prediction.

In [5]:
from torch_geometric.explain import Explainer, GNNExplainer


class TinyGCN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(16, 8)
        self.conv2 = GCNConv(8, 2)

    def forward(self, x, edge_index):
        return self.conv2(self.conv1(x, edge_index).relu(), edge_index)


explainer = Explainer(
    model=TinyGCN().to(device),
    algorithm=GNNExplainer(epochs=50),
    explanation_type="model",
    node_mask_type="attributes",
    edge_mask_type="object",
    model_config=dict(mode="multiclass_classification", task_level="node", return_type="raw"),
)
explanation = explainer(ppi_gpu.x, ppi_gpu.edge_index, index=0)
print("Edge importance for protein 0:", explanation.edge_mask.cpu().numpy().round(2))

Edge importance for protein 0: [0.66 0.59 0.   0.21 0.   0.  ]


## 3. Protein language model (ESM-2)

Uses the smallest ESM-2 model (8M parameters, ~30 MB download) only to test the pipeline.
The real protein embeddings will use a larger ESM-2 model.

In [6]:
import os

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"  # Windows: the model cache works fine without symlinks

from huggingface_hub.utils import logging as hub_logging
from transformers import AutoTokenizer, EsmModel
from transformers.utils import logging as hf_logging

hub_logging.set_verbosity_error()  # hide the "no HF token" notice: public models need no login
hf_logging.set_verbosity_error()   # hide the load report: the masked-LM head is unused, we only need the encoder

model_name = "facebook/esm2_t6_8M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_name)
esm = EsmModel.from_pretrained(model_name, add_pooling_layer=False).to(device).eval()

# Human insulin precursor (UniProt P01308), 110 amino acids
sequence = ("MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAED"
            "LQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN")

with torch.no_grad():
    tokens = tokenizer(sequence, return_tensors="pt").to(device)
    residue_emb = esm(**tokens).last_hidden_state[0, 1:-1]  # drop <cls> and <eos> tokens
    protein_emb = residue_emb.mean(dim=0)                   # one vector per protein

print("Residues          :", len(sequence))
print("Per-residue emb.  :", tuple(residue_emb.shape))
print("Protein embedding :", tuple(protein_emb.shape), protein_emb[:4].cpu().numpy().round(3), "...")

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 2321.26it/s]


Residues          : 110
Per-residue emb.  : (110, 320)
Protein embedding : (320,) [ 0.061 -0.349  0.209  0.257] ...


## 4. Other libraries

In [7]:
from importlib import import_module
from importlib.metadata import version

libraries = {  # import name -> pip package name
    "numpy": "numpy", "pandas": "pandas", "scipy": "scipy", "sklearn": "scikit-learn",
    "pyarrow": "pyarrow", "Bio": "biopython", "obonet": "obonet", "networkx": "networkx",
    "pyvis": "pyvis", "matplotlib": "matplotlib", "seaborn": "seaborn",
    "transformers": "transformers", "streamlit": "streamlit", "requests": "requests",
    "dotenv": "python-dotenv", "tqdm": "tqdm",
}
for module, package in libraries.items():
    import_module(module)
    print(f"  {package:14s} {version(package)}")

  numpy          2.4.6
  pandas         3.0.6
  scipy          1.17.1
  scikit-learn   1.9.1
  pyarrow        25.0.1
  biopython      1.88
  obonet         1.3.0
  networkx       3.6.1
  pyvis          0.3.2
  matplotlib     3.11.2
  seaborn        0.13.2
  transformers   5.19.0
  streamlit      1.65.0
  requests       2.34.2
  python-dotenv  1.2.4
  tqdm           4.70.1


## 5. Progress bars

Long steps (downloads, embeddings, training) show `tqdm` progress bars. They are plain text
on purpose: ipywidgets is not installed because VS Code's widget loader is unreliable.
A bar counting up to 50 should appear below this cell.

In [8]:
import time

from tqdm.auto import tqdm

for _ in tqdm(range(50), desc="Progress bar test"):
    time.sleep(0.02)

Progress bar test: 100%|██████████| 50/50 [00:01<00:00, 48.25it/s]


**All cells ran without errors and the progress bar appeared → the environment is ready.**